In [ ]:
# -*- coding: utf-8 -*-
"""Semantic search.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1ufNq5rxqT28fvLoi5-tYp8EPQD_7zYE6
"""

In [1]:
# Install sentence-transformers for semantic search and rank-bm25 for keyword search
%%capture
!pip install -q sentence-transformers rank-bm25

In [4]:
# Sample corporate/tech dataset
corpus = [
    "How to hard-reset your iPhone 13 if the touch screen is completely frozen or unresponsive.", # Doc 0
    "Troubleshooting guide for iOS updates failing on newer Apple mobile devices.",              # Doc 1
    "The new Samsung Galaxy S26 Ultra features an advanced generative AI camera system.",         # Doc 2
    "Steps to recover a lost Google Pixel account recovery phrase or authentication token.",    # Doc 3
    "Fixing Wi-Fi connectivity drops and network configuration errors on Apple Macbook laptops.", # Doc 4
    "Why is my smartphone battery draining so quickly? Top power optimization tips.",            # Doc 5
]

print(f"Loaded database with {len(corpus)} technical documents.")

Loaded database with 6 technical documents.


In [6]:
from rank_bm25 import BM25Okapi
import numpy as np

# Tokenize corpus for BM25 processing
tokenized_corpus = [doc.lower().split() for doc in corpus]

# Initialize BM25 model
bm25 = BM25Okapi(tokenized_corpus)

def sparse_search(query, top_n=3):
    """Executes keyword lookup and returns top document indices and raw scores."""

    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)

    # Sort indices by score (descending)
    top_indices = np.argsort(scores)[::-1][:top_n]

    # Return only documents with positive scores
    results = [(idx, scores[idx]) for idx in top_indices if scores[idx] > 0]

    return results

# Quick verification test
print("Sparse test search for 'iPhone 13':", sparse_search("iPhone 13"))

Sparse test search for 'iPhone 13': [(np.int64(0), np.float64(2.3996477957205307))]


In [11]:
from sentence_transformers import SentenceTransformer, util
import numpy as np

# Initialize model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Encode corpus
corpus_embeddings = embedding_model.encode(corpus, convert_to_tensor=True)

def dense_search(query, top_n=5):
    """Executes semantic search via cosine similarity and returns sorted indices and scores."""

    query_embedding = embedding_model.encode(query, convert_to_tensor=True)

    # Calculate cosine similarity against all documents
    cos_scores = util.cos_sim(query_embedding, corpus_embeddings)[0]

    # Sort top elements
    top_results = np.argsort(cos_scores.cpu().numpy())[::-1][:top_n]

    return [(int(idx), float(cos_scores[idx])) for idx in top_results]

# Quick verification test
print("Dense test search for 'pixels in camera':", dense_search("pixels in camera"))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Dense test search for 'pixels in camera': [(2, 0.23958879709243774), (3, 0.12979185581207275), (0, 0.1011519730091095), (5, 0.04306134581565857), (1, -0.06324774771928787)]


In [12]:
def reciprocal_rank_fusion(sparse_results, dense_results, k=60, top_n=3):
    """
    Fuses rankings from separate systems using the Reciprocal Rank Fusion formula.
    Inputs are expected to be lists of tuples: (doc_id, score) sorted by relevance.
    """
    rrf_scores = {}

    # Process sparse rankings
    for rank, (doc_id, _) in enumerate(sparse_results):
        # rank starts at 0, so rank position is rank + 1
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k + (rank + 1))

    # Process dense rankings
    for rank, (doc_id, _) in enumerate(dense_results):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k + (rank + 1))

    # Sort documents based on their combined RRF metrics
    fused_rankings = sorted(rrf_scores.items(), key=lambda item: item[1], reverse=True)

    return fused_rankings[:top_n]

In [13]:
def hybrid_search_engine(query, top_n=3):
    """Orchestrates both systems and fuses the output."""
    # 1. Gather deep recall arrays from both engines
    sparse_res = sparse_search(query, top_n=10)
    dense_res = dense_search(query, top_n=10)

    # 2. Fuse the rankings using RRF
    hybrid_res = reciprocal_rank_fusion(sparse_res, dense_res, k=60, top_n=top_n)

    # 3. Print pretty outputs for comparison
    print(f"\n==== TARGET QUERY: '{query}' ====")
    for rank, (doc_id, rrf_score) in enumerate(hybrid_res):
        print(f"\n[Rank {rank + 1}] (RRF Score: {rrf_score:.4f})")
        print(f"Document #{doc_id}: {corpus[doc_id]}")

In [16]:
# Execution Test 1: The Keyword Trap
# Query uses exact keyword terms from Doc 0, but conceptual theme of Doc 1
hybrid_search_engine("Apple mobile device issues")

# Execution Test 2: Multi-Concept Search
# Query matches semantic context of Doc 5 but specific tech hardware in Doc 4
hybrid_search_engine("Macbook laptop battery optimization")


==== TARGET QUERY: 'Apple mobile device issues' ====

[Rank 1] (RRF Score: 0.0328)
Document #1: Troubleshooting guide for iOS updates failing on newer Apple mobile devices.

[Rank 2] (RRF Score: 0.0318)
Document #4: Fixing Wi-Fi connectivity drops and network configuration errors on Apple Macbook laptops.

[Rank 3] (RRF Score: 0.0161)
Document #5: Why is my smartphone battery draining so quickly? Top power optimization tips.

==== TARGET QUERY: 'Macbook laptop battery optimization' ====

[Rank 1] (RRF Score: 0.0328)
Document #5: Why is my smartphone battery draining so quickly? Top power optimization tips.

[Rank 2] (RRF Score: 0.0323)
Document #4: Fixing Wi-Fi connectivity drops and network configuration errors on Apple Macbook laptops.

[Rank 3] (RRF Score: 0.0159)
Document #2: The new Samsung Galaxy S26 Ultra features an advanced generative AI camera system.


In [26]:
def hybrid_search_engine(query, top_n=3):

    sparse_res = sparse_search(query, top_n=10)
    dense_res = dense_search(query, top_n=10)

    results = weighted_reciprocal_rank_fusion(
        sparse_res,
        dense_res,
        sparse_weight=0.6,
        dense_weight=0.4,
        top_n=top_n
    )

    print(f"\n==== QUERY: {query} ====")

    for rank, (doc_id, score) in enumerate(results):
        print(f"[Rank {rank+1}] Doc #{doc_id} (score={score:.4f})")

    # THIS IS REQUIRED
    return results

In [27]:
import math

ground_truth = [
    {
        "query": "Apple mobile device issues",
        "relevant_docs": [1]
    },
    {
        "query": "Macbook laptop battery optimization",
        "relevant_docs": [4, 5]
    }
]

print("\n GROUND TRUTH DATA LOADED\n")

for item in ground_truth:
    print(f"Query: {item['query']}")
    print(f"Relevant Docs: {item['relevant_docs']}")
    print("-" * 40)


 GROUND TRUTH DATA LOADED

Query: Apple mobile device issues
Relevant Docs: [1]
----------------------------------------
Query: Macbook laptop battery optimization
Relevant Docs: [4, 5]
----------------------------------------


In [28]:
#Metrices
#DCG
def dcg(relevances):
    return sum(rel / math.log2(i + 2) for i, rel in enumerate(relevances))

In [29]:
#nDCG
def ndcg(retrieved_docs, relevant_docs, k=3):

    # Actual ranking
    relevances = [1 if doc in relevant_docs else 0 for doc in retrieved_docs[:k]]
    actual_dcg = dcg(relevances)

    # Ideal ranking (BEST possible ordering)
    ideal_relevances = sorted([1]*len(relevant_docs) + [0]*k, reverse=True)[:k]
    ideal_dcg = dcg(ideal_relevances)

    return actual_dcg / ideal_dcg if ideal_dcg > 0 else 0.0

In [30]:
#MRR
def mrr(retrieved_docs, relevant_docs):
    for i, doc in enumerate(retrieved_docs):
        if doc in relevant_docs:
            return 1 / (i + 1)
    return 0.0

In [32]:
#HYBRID SEARCH EVALUATOR

def evaluate_hybrid_search(search_function, ground_truth, k=3):

    print("\n=======================================")
    print("HYBRID SEARCH EVALUATION STARTED")
    print("=======================================\n")

    all_mrr = []
    all_ndcg = []

    for item in ground_truth:

        query = item["query"]
        relevant_docs = item["relevant_docs"]

        print("\n---------------------------------------")
        print(f" QUERY: {query}")
        print(f" GROUND TRUTH: {relevant_docs}")

        # Run search
        results = search_function(query, top_n=k)

        #  Safety check (IMPORTANT FIX)
        if results is None:
            print("Warning: search returned None")
            continue

        retrieved_docs = [doc_id for doc_id, _ in results]

        print(f"RETRIEVED DOCS: {retrieved_docs}")

        # Metrics
        score_mrr = mrr(retrieved_docs, relevant_docs)
        score_ndcg = ndcg(retrieved_docs, relevant_docs, k)

        print(f"MRR: {score_mrr:.4f}")
        print(f"nDCG@{k}: {score_ndcg:.4f}")

        all_mrr.append(score_mrr)
        all_ndcg.append(score_ndcg)

    # Final averages
    avg_mrr = sum(all_mrr) / len(all_mrr)
    avg_ndcg = sum(all_ndcg) / len(all_ndcg)

    print("\n=======================================")
    print("FINAL EVALUATION RESULTS")
    print("=======================================")
    print(f"Average MRR   : {avg_mrr:.4f}")
    print(f"Average nDCG@{k}: {avg_ndcg:.4f}")
    print("=======================================\n")

    return avg_mrr, avg_ndcg

In [33]:
avg_mrr, avg_ndcg = evaluate_hybrid_search(
    hybrid_search_engine,
    ground_truth,
    k=3
)


HYBRID SEARCH EVALUATION STARTED


---------------------------------------
 QUERY: Apple mobile device issues
 GROUND TRUTH: [1]

==== QUERY: Apple mobile device issues ====
[Rank 1] Doc #1 (score=0.0164)
[Rank 2] Doc #4 (score=0.0159)
[Rank 3] Doc #5 (score=0.0065)
RETRIEVED DOCS: [np.int64(1), np.int64(4), 5]
MRR: 1.0000
nDCG@3: 1.0000

---------------------------------------
 QUERY: Macbook laptop battery optimization
 GROUND TRUTH: [4, 5]

==== QUERY: Macbook laptop battery optimization ====
[Rank 1] Doc #5 (score=0.0164)
[Rank 2] Doc #4 (score=0.0161)
[Rank 3] Doc #2 (score=0.0063)
RETRIEVED DOCS: [np.int64(5), np.int64(4), 2]
MRR: 1.0000
nDCG@3: 1.0000

FINAL EVALUATION RESULTS
Average MRR   : 1.0000
Average nDCG@3: 1.0000

